# AWREAS Revision Quality Scorer
**Metrics:** ROUGE-L · BERTScore · Improvement Delta

This notebook evaluates the quality of the AWREAS Revision Engine against
ArgRewrite V.2 reference revisions. For each essay pair it:
1. Submits the **Draft 2** text to the `/review-revise` endpoint.
2. Compares the **system-generated revision** to the **Draft 3** reference using ROUGE-L and BERTScore.
3. Computes an **Improvement Delta** (score of revised draft minus score of original).

**Before running:** Make sure your project zip is uploaded to Google Drive.

## Cell 1 — Mount Drive and Extract Project

In [2]:
from google.colab import drive
import os, sys, glob

drive.mount('/content/drive')

zip_path   = '/content/drive/MyDrive/academic-review-system.zip'
extract_dir = '/content/academic-review-system'

if os.path.exists(zip_path):
    print('Found zip — extracting...')
    !unzip -q -o "{zip_path}" -d "{extract_dir}"
    print('Done.')
else:
    extract_dir = '/content/drive/MyDrive/academic-review-system'
    if not os.path.exists(extract_dir):
        raise FileNotFoundError('Upload academic-review-system.zip to Google Drive first.')

toml_paths = glob.glob(os.path.join(extract_dir, '**/pyproject.toml'), recursive=True)
if not toml_paths:
    raise FileNotFoundError('Cannot locate pyproject.toml inside extracted files.')

project_root = os.path.dirname(toml_paths[0])
print(f'Project root: {project_root}')
sys.path.insert(0, project_root)
%cd "{project_root}"

Mounted at /content/drive
Found zip — extracting...
Done.
Project root: /content/academic-review-system
/content/academic-review-system


## Cell 2 — Install Dependencies

In [3]:
# Core project dependencies
!pip install -q .

# Revision quality metric libraries
!pip install -q rouge-score bert-score

# Google GenAI SDK (needed even if you use OpenAI/OpenRouter, for clean imports)
!pip install -q google-genai

print('All dependencies installed.')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 48.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 66.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.3/77.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 82.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.3/346.3 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Cell 3 — Configure LLM Provider and API Key

In [4]:
import getpass

print('Available providers: gemini | openrouter | openai | qwen')
provider = input('Enter LLM provider: ').strip().lower()
if provider not in ('gemini', 'openrouter', 'openai', 'qwen'):
    print("Unrecognised provider — defaulting to 'openai'.")
    provider = 'openai'

api_key = getpass.getpass(f'Paste your {provider.upper()} API key: ').strip()
if not api_key:
    raise ValueError('API key is required.')

# For OpenAI, gpt-4o-mini is recommended (cheap + capable enough for revision)
MODEL_SUGGESTIONS = {
    'openai':     'gpt-4o-mini',
    'openrouter': 'anthropic/claude-sonnet-4.6',
    'gemini':     'gemini-2.5-flash',
    'qwen':       'qwen3.7-max',
}
default_model = MODEL_SUGGESTIONS[provider]
model_input = input(f'Model name [{default_model}]: ').strip()
model_name = model_input if model_input else default_model

print(f'\nProvider : {provider}')
print(f'Model    : {model_name}')
print('Ready.')

Available providers: gemini | openrouter | openai | qwen

Provider : openai
Model    : gpt-4o-mini
Ready.


## Cell 4 — Initialise AWREAS (LLMClient + SupervisorAgent)

In [5]:
# Clear cached imports so Colab picks up freshly extracted files
for mod in [k for k in sys.modules if k.startswith('app')]:
    del sys.modules[mod]

from app.core.config import settings
settings.ENABLE_WEB_RESEARCH = False   # keeps evaluation fast; no external lookups
settings.LLM_REQUEST_TIMEOUT = 90.0    # revisions take longer than scoring-only calls

from app.llm.client import LLMClient
from app.supervisor.agent import SupervisorAgent
from app.services.revision import RevisionService

llm_client = LLMClient(api_key=api_key, provider=provider, default_model=model_name)
supervisor = SupervisorAgent(llm_client=llm_client)

# Point all model slots at the same chosen model so every component is consistent
settings.WORKER_MODEL_NAME    = 'gpt-4o-mini'
settings.SYNTHESIS_MODEL_NAME = 'gpt-4o-mini'
settings.REVISION_MODEL_NAME  = 'gpt-4o-mini'
settings.REVIEW_MODEL_NAME    = 'gpt-4o-mini'

print(f'SupervisorAgent ready (provider={provider}, model={model_name}).')

{"timestamp": "2026-06-10 12:45:48,307", "level": "INFO", "correlation_id": "N/A", "service": "academic_review_service", "message": "LLM client initialised (provider=gemini, model=anthropic/claude-sonnet-4.6).", "module": "client", "function": "_init_client"}
SupervisorAgent ready (provider=openai, model=gpt-4o-mini).


## Cell 5 — Download and Prepare ArgRewrite V.2 Corpus

ArgRewrite V.2 (Kashefi et al., 2022) contains three successive drafts of
argumentative essays on self-driving cars. We use:
- **Draft 2** as the input submitted to the Revision Engine.
- **Draft 3** as the human reference revision we compare against.

The dataset is hosted on GitHub and downloaded directly.

In [ ]:
import requests, json, os
from pathlib import Path

ARGREWRITE_URL = (
    'https://raw.githubusercontent.com/tuveri/argRewrite/master/data/argRewrite_v2.json'
)
CORPUS_PATH = Path('data/argrewrite/argRewrite_v2.json')
CORPUS_PATH.parent.mkdir(parents=True, exist_ok=True)

if not CORPUS_PATH.exists():
    print('Downloading ArgRewrite V.2...')
    r = requests.get(ARGREWRITE_URL, timeout=30)
    if r.status_code == 200:
        CORPUS_PATH.write_bytes(r.content)
        print(f'Saved to {CORPUS_PATH}')
    else:
        # Fallback: use the ACL Anthology direct link
        ARGREWRITE_FALLBACK = 'https://aclanthology.org/2022.lrec-1.97.pdf'
        print(f'Download failed (HTTP {r.status_code}).')
        print('Please manually download ArgRewrite V.2 from:')
        print('  https://github.com/tuveri/argRewrite')
        print('and upload argRewrite_v2.json to your Google Drive, then copy it here:')
        print(f'  {CORPUS_PATH}')
        raise RuntimeError('ArgRewrite download failed — see instructions above.')
else:
    print(f'ArgRewrite corpus already present at {CORPUS_PATH}')

# ---------------------------------------------------------------
# Parse into (draft2_text, draft3_text) pairs.
# The JSON structure is: list of participants, each with 'drafts'
# keyed by '1', '2', '3'. The text field is 'essay' or 'text'.
# We handle both known schema variants here.
# ---------------------------------------------------------------
raw = json.loads(CORPUS_PATH.read_text(encoding='utf-8'))

pairs = []  # list of {"id": str, "draft2": str, "draft3": str}

def _extract_text(obj):
    """Pull the essay text out regardless of key name."""
    for key in ('essay', 'text', 'content', 'body'):
        if key in obj and isinstance(obj[key], str) and obj[key].strip():
            return obj[key].strip()
    # Some versions store sentences as a list — join them
    for key in ('sentences', 'sents'):
        if key in obj and isinstance(obj[key], list):
            return ' '.join(s.get('text', s) if isinstance(s, dict) else s
                            for s in obj[key]).strip()
    return None

entries = raw if isinstance(raw, list) else raw.get('data', raw.get('essays', []))

for entry in entries:
    participant_id = str(entry.get('id', entry.get('participant_id', len(pairs))))
    drafts = entry.get('drafts', {})

    # Handle both integer and string keys
    d2 = drafts.get(2) or drafts.get('2') or drafts.get('draft2')
    d3 = drafts.get(3) or drafts.get('3') or drafts.get('draft3')

    if d2 is None or d3 is None:
        continue  # skip incomplete entries

    t2 = _extract_text(d2) if isinstance(d2, dict) else (d2.strip() if isinstance(d2, str) else None)
    t3 = _extract_text(d3) if isinstance(d3, dict) else (d3.strip() if isinstance(d3, str) else None)

    if t2 and t3 and len(t2) >= 100:
        pairs.append({'id': participant_id, 'draft2': t2, 'draft3': t3})

print(f'Loaded {len(pairs)} essay pairs with Draft 2 + Draft 3.')
if pairs:
    sample = pairs[0]
    print(f'\nSample ID : {sample["id"]}')
    print(f'Draft 2   : {sample["draft2"][:120]}...')
    print(f'Draft 3   : {sample["draft3"][:120]}...')

## Cell 6 — Sampling Configuration

The full corpus has ~60 participants × 2 usable pairs each.
For thesis evaluation 30 pairs (stratified across essay quality) is standard.
Adjust `SAMPLE_SIZE` below.

In [ ]:
import random

# ---------------------------------------------------------------
# CONFIGURATION — adjust as needed
# ---------------------------------------------------------------
SAMPLE_SIZE     = 30       # Set to None to run all pairs
RANDOM_SEED     = 42       # For reproducibility
CITATION_STYLE  = 'harvard'
# ---------------------------------------------------------------

random.seed(RANDOM_SEED)

if SAMPLE_SIZE is not None and len(pairs) > SAMPLE_SIZE:
    sample_pairs = random.sample(pairs, SAMPLE_SIZE)
else:
    sample_pairs = pairs

print(f'Running evaluation on {len(sample_pairs)} essay pairs.')
print(f'Citation style : {CITATION_STYLE}')
print(f'Random seed    : {RANDOM_SEED}')

## Cell 7 — Run AWREAS Revision Engine on All Pairs

Each Draft 2 essay is submitted to the full supervisor pipeline
(all 8 worker agents + Revision Engine). The `revised_content` field
from the response is stored alongside the reference Draft 3.

In [ ]:
import asyncio, time
from app.services.workflow import ReviewWorkflowService
from app.schemas.review import EvaluationRequestSchema

revision_records = []  # {id, draft2, draft3, revised, original_score, revised_score}
failed_ids = []

total = len(sample_pairs)
print(f'Starting revision evaluation on {total} essays...\n')

for i, pair in enumerate(sample_pairs):
    essay_id = pair['id']
    draft2   = pair['draft2']
    draft3   = pair['draft3']

    try:
        start = time.perf_counter()

        # --- Step A: Review + Revise ---
        rr_request = EvaluationRequestSchema(
            content=draft2,
            citation_style=CITATION_STYLE,
            use_llm_rewrite=True,
        )
        rr_result = await ReviewWorkflowService.review_revise_text(
            request=rr_request,
            supervisor=supervisor,
            session=None,   # no DB needed for offline evaluation
        )
        original_score = rr_result.overall_score
        revised_text   = rr_result.revised_content

        # --- Step B: Re-evaluate the revised text to get its score ---
        eval_request = EvaluationRequestSchema(
            content=revised_text,
            citation_style=CITATION_STYLE,
        )
        eval_result = await ReviewWorkflowService.evaluate_text(
            request=eval_request,
            supervisor=supervisor,
        )
        revised_score = eval_result.overall_score

        elapsed = (time.perf_counter() - start) * 1000

        revision_records.append({
            'id':             essay_id,
            'draft2':         draft2,
            'draft3':         draft3,          # human reference
            'revised':        revised_text,    # system-generated revision
            'original_score': original_score,
            'revised_score':  revised_score,
            'revision_mode':  rr_result.revision_mode,
            'processing_ms':  elapsed,
        })
        print(f'[{i+1}/{total}] ID={essay_id} | orig={original_score:.1f} '
              f'rev={revised_score:.1f} | mode={rr_result.revision_mode} '
              f'| {elapsed:.0f}ms')

    except Exception as exc:
        print(f'[{i+1}/{total}] ID={essay_id} FAILED: {exc}')
        failed_ids.append(essay_id)

print(f'\nCompleted: {len(revision_records)} succeeded, {len(failed_ids)} failed.')
if failed_ids:
    print(f'Failed IDs: {failed_ids}')

## Cell 8 — Compute ROUGE-L

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

rouge_scores = []
for rec in revision_records:
    result = scorer.score(
        target=rec['draft3'],      # reference: what the human produced
        prediction=rec['revised'], # hypothesis: what AWREAS produced
    )
    f1 = result['rougeL'].fmeasure
    rouge_scores.append(f1)
    rec['rouge_l'] = f1

mean_rouge = sum(rouge_scores) / len(rouge_scores)
print(f'ROUGE-L F1  —  mean={mean_rouge:.4f}  '
      f'min={min(rouge_scores):.4f}  max={max(rouge_scores):.4f}')

## Cell 9 — Compute BERTScore

In [ ]:
from bert_score import score as bert_score

hypotheses  = [rec['revised'] for rec in revision_records]
references  = [rec['draft3']  for rec in revision_records]

print('Computing BERTScore (this downloads a model on first run — ~400MB)...')
P, R, F1 = bert_score(
    cands=hypotheses,
    refs=references,
    lang='en',
    model_type='roberta-large',  # recommended for English text evaluation
    verbose=True,
)

bert_f1_list = F1.tolist()
for rec, f1 in zip(revision_records, bert_f1_list):
    rec['bertscore_f1'] = f1

mean_bert = sum(bert_f1_list) / len(bert_f1_list)
print(f'\nBERTScore F1  —  mean={mean_bert:.4f}  '
      f'min={min(bert_f1_list):.4f}  max={max(bert_f1_list):.4f}')

## Cell 10 — Compute Improvement Delta

In [ ]:
deltas = []
positive_deltas = 0

for rec in revision_records:
    delta = rec['revised_score'] - rec['original_score']
    rec['improvement_delta'] = round(delta, 4)
    deltas.append(delta)
    if delta > 0:
        positive_deltas += 1

mean_delta   = sum(deltas) / len(deltas)
pct_positive = positive_deltas / len(deltas) * 100

print(f'Improvement Delta  —  mean={mean_delta:+.4f}  '
      f'min={min(deltas):+.4f}  max={max(deltas):+.4f}')
print(f'Essays with positive delta: {positive_deltas}/{len(deltas)} ({pct_positive:.1f}%)')

## Cell 11 — Summary Results Table

In [ ]:
import statistics

rouge_vals  = [r['rouge_l']           for r in revision_records]
bert_vals   = [r['bertscore_f1']      for r in revision_records]
delta_vals  = [r['improvement_delta'] for r in revision_records]

def fmt(vals):
    return (
        f"mean={statistics.mean(vals):.4f}  "
        f"median={statistics.median(vals):.4f}  "
        f"stdev={statistics.stdev(vals):.4f}  "
        f"min={min(vals):.4f}  max={max(vals):.4f}"
    )

print('=' * 60)
print('AWREAS REVISION QUALITY EVALUATION SUMMARY')
print('=' * 60)
print(f'Corpus         : ArgRewrite V.2 (Kashefi et al., 2022)')
print(f'Pairs evaluated: {len(revision_records)}')
print(f'Provider/Model : {provider} / {model_name}')
print()
print(f'ROUGE-L F1     : {fmt(rouge_vals)}')
print(f'BERTScore F1   : {fmt(bert_vals)}')
print(f'Improv. Delta  : {fmt(delta_vals)}')
print(f'  Positive     : {positive_deltas}/{len(deltas)} ({pct_positive:.1f}%)')
print('=' * 60)

## Cell 12 — Save Results to JSON

In [ ]:
import json
from pathlib import Path

output_path = Path('data/argrewrite/revision_quality_results.json')
output_path.parent.mkdir(parents=True, exist_ok=True)

summary = {
    'meta': {
        'corpus':       'ArgRewrite V.2',
        'reference':    'Kashefi et al. (2022)',
        'pairs':        len(revision_records),
        'provider':     provider,
        'model':        model_name,
        'sample_size':  SAMPLE_SIZE,
        'random_seed':  RANDOM_SEED,
    },
    'aggregate': {
        'rouge_l_mean':          round(statistics.mean(rouge_vals), 4),
        'rouge_l_stdev':         round(statistics.stdev(rouge_vals), 4),
        'bertscore_f1_mean':     round(statistics.mean(bert_vals), 4),
        'bertscore_f1_stdev':    round(statistics.stdev(bert_vals), 4),
        'improvement_delta_mean': round(statistics.mean(delta_vals), 4),
        'improvement_delta_stdev': round(statistics.stdev(delta_vals), 4),
        'pct_positive_delta':    round(pct_positive, 2),
    },
    'per_essay': [
        {
            'id':               r['id'],
            'original_score':   r['original_score'],
            'revised_score':    r['revised_score'],
            'improvement_delta': r['improvement_delta'],
            'rouge_l':          round(r['rouge_l'], 4),
            'bertscore_f1':     round(r['bertscore_f1'], 4),
            'revision_mode':    r['revision_mode'],
            'processing_ms':    round(r['processing_ms'], 0),
        }
        for r in revision_records
    ],
}

output_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(f'Results saved to: {output_path}')

## Cell 13 — Back Up Results to Google Drive

In [ ]:
import shutil

drive_dest = '/content/drive/MyDrive/AWREAS_Revision_Results'
os.makedirs(drive_dest, exist_ok=True)

dest = shutil.copy(str(output_path), drive_dest)
print(f'Backed up to Google Drive: {dest}')